<a href="https://colab.research.google.com/github/9marlon9/ICAD/blob/main/AED_Ejemplo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Marlon Angulo
# Importar la biblioteca de programación lineal
from pulp import *

# Definir personas
personas = [
    {'nombre': 'Ana', 'x': 5, 'y': 8},      # Int=5, Usos=8
    {'nombre': 'Juan', 'x': 2, 'y': 3},     # Int=2, Usos=3
    {'nombre': 'Carlos', 'x': 6, 'y': 4}
]

N = len(personas)  # Número de personas

# Función para resolver el DEA para una persona específica (índice k)
def calcular_dea(k):

    prob = LpProblem(f"DEA_para_{personas[k]['nombre']}", LpMaximize)

    # Variables: pesos wu (usos) y wi (intenciones)
    wu = LpVariable(f"wu_{k}", lowBound=0)  # >=0
    wi = LpVariable(f"wi_{k}", lowBound=0)  # >=0

    # Función objetivo: maximizar θ_k = wu * y_k + wi * x_k
    prob += wu * personas[k]['y'] + wi * personas[k]['x']

    # Restricciones: para cada persona j (incluyendo k), wu * y_j + wi * x_j <= 1
    for j in range(N):
        prob += wu * personas[j]['y'] + wi * personas[j]['x'] <= 1, f"Restriccion_{j}_para_{k}"

    # Resolver el problema
    prob.solve(PULP_CBC_CMD(msg=0))  # msg=0 para no mostrar logs verbose

    # Obtener resultados
    theta = value(prob.objective)
    wu_opt = value(wu)
    wi_opt = value(wi)

    return theta, wu_opt, wi_opt

# Calcular para cada persona
resultados = []
for k in range(N):
    theta, wu, wi = calcular_dea(k)
    resultados.append({
        'Persona': personas[k]['nombre'],
        'Índice θ': round(theta, 4),
        'Peso Usos (wu)': round(wu, 4),
        'Peso Intenciones (wi)': round(wi, 4)
    })

# Imprimir resultados en tabla
for res in resultados:
    print(f"| {res['Persona']:7} | {res['Índice θ']:8} | {res['Peso Usos (wu)']:14} | {res['Peso Intenciones (wi)']:21} |")


| Ana     |      1.0 |         0.0357 |                0.1429 |
| Juan    |   0.3929 |         0.0357 |                0.1429 |
| Carlos  |      1.0 |            0.0 |                0.1667 |
